# Closed-Loop Gain Recovery Test

Synthetic closed-loop demonstration of the `newnucal` calibration pipeline:

1. Build a HERA-like array and a random chromatic sky model
2. Simulate visibilities with `ForwardModel` at several times (Earth rotation) and frequencies
3. Apply known per-frequency gain degeneracies (amplitude, phase, phase gradient)
4. Recover gains with the sky held fixed — verifying the solution reproduces the data
5. Show joint sky + gain recovery starting from a perturbed sky

The key physics being tested: the DPSS spectral constraints on the sky/beam model, combined with Earth rotation that decorrelates sky pixels from beam pixels across time, provide enough information to reduce the residuals.  Note that the recovered gains need not match the injected ones exactly — there are residual degeneracies — but the calibrated model visibilities should match the data.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u
import healpy

jax.config.update("jax_enable_x64", False)  # float32 throughout

from newnucal import HERAArray, BeamModel, BeamBasis, SkyBasis, SkyModel, ForwardModel, Calibrator, apply_gains, init_gain_params
from newnucal.basis import BeamBasis, SkyBasis
#from newnucal.grid_fitter import GridFitter
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 1. Array, frequency, and time setup

In [ ]:

# --- Array ---
array = HERAArray.from_hex(hexnum=5, sep=14.6)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

# --- Frequencies ---
#freqs = array.freq_array(50e6, 225e6)  # Hz
#nfreq = freqs.size
nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq, dtype=np.float32)  # Hz
print(f"Freqs: {nfreq}, ({freqs[0]/1e6}--{freqs[-1]/1e6} MHz)")

# --- Times ---
hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
nhrs = 2
ntime = nhrs * 4
dt = nhrs * 60  * u.min / (ntime + 1)
times = t0 + np.arange(ntime) * dt
print(f"Times: {ntime},  span: {(times[-1] - times[0]).to(u.min):.1f}")

# --- Rotation matrices ---
rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")


## 2. Beam and sky model

In [ ]:
sky_nside = 32
beam_nside = 16

# Beam: Airy disk (HERA dish diameter 14.6 m), DPSS eta_max = 20 ns
#beam_basis = BeamBasis.from_dpss(freqs, eta_max=20e-9)
beam_basis = BeamBasis.from_file('beam_basis_airy.npz', new_freqs=freqs)
beam_model = BeamModel(nside=beam_nside, freqs=freqs, basis=beam_basis)
print(f"Beam Nmodes: {beam_model.nmodes}")

# Sky: random power-law HEALPix map
npix_sky = healpy.nside2npix(sky_nside)
rng = np.random.default_rng(42)

# Build a smooth chromatic sky: per-pixel spectral index drawn from N(-0.7, 0.1)
ref_freq = 150e6
spectral_indices = rng.normal(-0.7, 0.1, npix_sky).astype(np.float32)
ref_flux = rng.exponential(scale=1.0, size=npix_sky).astype(np.float32)
# flux_true[npix, nfreq]
flux_true = ref_flux[:, None] * (freqs[None, :] / ref_freq) ** spectral_indices[:, None]

# Project onto sky DPSS basis
#sky_basis = SkyBasis.from_dpss(freqs, eta_max=40e-9)
sky_basis = SkyBasis.from_file('sky_basis_gsm_gleam.npz', new_freqs=freqs)
sky_coeffs_true = jnp.array(sky_basis.project(flux_true), dtype=jnp.float32)
sky_model  = SkyModel(nside=sky_nside, freqs=freqs, basis=sky_basis)
print(f"Sky Nmodes: {sky_model.nmodes}")

## 3. Simulate true visibilities and apply known gain perturbations

In [ ]:
fwd = ForwardModel(array, sky_model, beam_model, freqs, eps=1e-5)

print("Simulating true visibilities (this compiles the JIT on first run)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape},  dtype: {vis_true.dtype}")
print(f"Mean |vis|: {float(jnp.abs(vis_true).mean()):.4f}")

In [ ]:
# --- True gain perturbations (per time and frequency) ---
# Gains vary smoothly in both time and frequency, mimicking slow ionospheric drift.

t_norm = np.linspace(0, 1, ntime)[:, None]   # (ntime, 1)  — normalised time axis
f_norm = np.linspace(0, 1, nfreq)[None, :]   # (1, nfreq) — normalised frequency axis

# log_amp: ~5% amplitude variation across band, with ~2% slow drift in time
true_log_amp = (
    0.05 * np.cos(2 * np.pi * f_norm)
    + 0.02 * np.sin(2 * np.pi * t_norm)
).astype(np.float32)  # (ntime, nfreq)

# phase: smooth variation in frequency, slow drift in time
true_phase = (
    0.15 * np.sin(2 * np.pi * f_norm)
    + 0.05 * np.cos(2 * np.pi * t_norm)
).astype(np.float32)  # (ntime, nfreq)

# phi: small phase gradients with independent time and frequency variation
true_phi = np.zeros((ntime, 2, nfreq), dtype=np.float32)
true_phi[:, 0, :] = (                               # East
    1e-4 * np.cos(2 * np.pi * f_norm)
    + 3e-5 * np.sin(2 * np.pi * t_norm)
)
true_phi[:, 1, :] = (                               # North
    5e-5 * np.sin(2 * np.pi * f_norm)
    + 2e-5 * np.cos(2 * np.pi * t_norm)
)

true_log_amp_j = jnp.array(true_log_amp)
true_phase_j   = jnp.array(true_phase)
true_phi_j     = jnp.array(true_phi)

vis_data = apply_gains(vis_true, true_log_amp_j, true_phase_j, true_phi_j,
                       jnp.array(array.bls, dtype=jnp.float32))

print(f"vis_data shape: {vis_data.shape}")
print(f"log_amp RMS across times: {float(jnp.std(jnp.array(true_log_amp), axis=0).mean()):.4f}")
print(f"phi_E  RMS across times:  {float(jnp.std(jnp.array(true_phi[:, 0, :]), axis=0).mean()):.2e}")

In [ ]:
cal = Calibrator(
    array=array,
    sky_model=sky_model,
    beam_model=beam_model,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

In [ ]:
#bm_wgts = cal.get_sky_beam_weighting()
#threshold = bm_wgts.max() / 100
#sky_mask = bm_wgts > threshold
#cal.apply_sky_mask(sky_mask)
#beam_mask = cal.build_beam_mask_from_sky_pixels(sky_mask)
beam_mask = cal.build_beam_mask_altitude(60)
sky_mask = cal.build_sky_mask_from_beam_pixels(beam_mask)
cal.apply_sky_mask(sky_mask)
cal.apply_beam_mask(beam_mask)

In [ ]:
fig = plt.figure()
#healpy.mollview(bm_wgts, fig=fig, sub=(2, 1, 1), title="Sky Beam Weighting", cmap="plasma")
#healpy.mollview(sky_mask,fig=fig, sub=(2, 1, 2), title="Sky Pixel Mask", cmap="gray")
healpy.mollview(sky_mask,fig=fig, title="Sky Pixel Mask", cmap="gray")

In [ ]:
fig = plt.figure()
healpy.orthview(beam_mask, rot=(0, 90, 0), half_sky=True, fig=fig, title="Beam Mask", cmap="gray")

## 5. Data reproduction: baseline spectra and loss

In [ ]:
freq_mhz = freqs / 1e6
t_show = 0
bls_j    = jnp.array(array.bls, dtype=jnp.float32)
beam_j   = jnp.array(beam_model.coeffs, dtype=jnp.float32)
bl_indices = [0, array.nbls // 4, array.nbls // 2, array.nbls - 1]

## 6. Joint sky + gain + beam recovery (perturbed start)

Start the sky from a perturbed version of the truth (30% Gaussian noise on DPSS coefficients) and the beam from a perturbed version (10% noise). Jointly optimize sky, gains, and beam using `fit_alternating_dirty` with `solve_every` scheduling. This tests whether Earth rotation + DPSS spectral constraints are sufficient to disentangle all three components.

In [ ]:
rng2 = np.random.default_rng(99)
sky_coeffs_perturbed = sky_coeffs_true + 0.30 * jnp.array(
    rng2.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

# Perturb the beam coefficients by 10% Gaussian noise
beam_coeffs_true = jnp.array(beam_model.coeffs, dtype=jnp.float32)
rng3 = np.random.default_rng(77)
beam_coeffs_perturbed = beam_coeffs_true + 0.30 * jnp.array(
    rng3.standard_normal(beam_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(beam_coeffs_true).mean())

print(f"Sky perturbation RMS / signal RMS: "
      f"{float(jnp.std(sky_coeffs_perturbed - sky_coeffs_true)) / float(jnp.std(sky_coeffs_true)):.2f}")
print(f"Beam perturbation RMS / signal RMS: "
      f"{float(jnp.std(beam_coeffs_perturbed - beam_coeffs_true)) / float(jnp.std(beam_coeffs_true)):.2f}")

prms0 = {
    "sky_coeffs":  sky_coeffs_perturbed,
    "beam_coeffs": beam_coeffs_perturbed,
    **init_gain_params(ntime, nfreq),
}

In [ ]:
# First, inject RFI in one frequency channel and create test data
# Inject narrowband RFI in channel 20 (strong contamination, all times/baselines)
rfi_channel = 20
rfi_amplitude = 10.0  # 10x typical signal amplitude
vis_with_rfi = np.array(vis_data, copy=True)
vis_with_rfi[:, rfi_channel, :] += rfi_amplitude * (1.0 + 0.3j)  # Add complex RFI

# Switch calibrator to use RFI-contaminated data
cal.data = jnp.array(vis_with_rfi, dtype=jnp.complex64)

print(f"Injected RFI in channel {rfi_channel} ({freqs[rfi_channel]/1e6:.1f} MHz): amplitude = {rfi_amplitude:.1f}x signal")
print(f"Data RMS (all freqs): {float(jnp.abs(cal.data).mean()):.4f}")
print(f"Data RMS (RFI channel): {float(jnp.abs(cal.data[:, rfi_channel, :]).mean()):.4f}")


In [ ]:
from newnucal.rfi import RFIConfig

# Baseline: fit WITHOUT RFI weighting on contaminated data
print("\n" + "="*70)
print("BASELINE FIT (without RFI weighting on contaminated data)")
print("="*70)
prms_baseline, loss_baseline = cal.fit_joint_sky_beam_dirty(
    prms0.copy(),
    n_iter=30,
    sky_beam_reg=1e-3,
    solve_every={'gains': 1},  # Update gains every iteration, no RFI
    joint_anderson_history=2,
    verbose=False,
)
print(f"Baseline loss: {loss_baseline:.4e}")

# Now fit WITH RFI weighting enabled
print("\n" + "="*70)
print("RFI-WEIGHTED FIT (with soft weight optimization)")
print("="*70)

# Configure RFI reweighting: quadratic regularization with soft weight fitting
rfi_cfg = RFIConfig(
    min_weight=0.01,
    max_weight=1.0,
    smooth_window=5,
    score_center=1.5,
    score_slope=2.0,
    prior_power=1.0,
    model_power=1.0,
    blend=0.0,
    time_reduce="median",
    use_soft_weight_fit=True,        # Enable soft weight optimization
    regularization=1.0,               # λ for quadratic penalty
    regularization_power=2.0,         # Use quadratic (closed-form solution)
)

# Create initial flags and weights: all unflagged, equal prior
initial_flags = np.zeros((nfreq,), dtype=bool)
initial_channel_weights = np.ones((nfreq,), dtype=np.float32)

# Fit with RFI weighting: update weights every 4 iterations
prms_with_rfi, loss_with_rfi = cal.fit_joint_sky_beam_dirty(
    prms0.copy(),
    n_iter=30,
    sky_beam_reg=1e-3,
    solve_every={'gains': 1, 'rfi': 4},  # Update RFI weights every 4 iterations
    rfi_config=rfi_cfg,
    initial_flags=initial_flags,
    initial_channel_weights=initial_channel_weights,
    max_rfi_updates=None,  # Allow updates throughout fit
    joint_anderson_history=2,
    verbose=False,
)
print(f"RFI-weighted loss: {loss_with_rfi:.4e}")

# Extract final channel weights from state (if available)
# For now, we'll compute weights by running once more with diagnostics
print("\n" + "="*70)
print("RFI WEIGHT SUMMARY")
print("="*70)
print(f"Loss improvement with RFI weighting: {loss_baseline/loss_with_rfi:.2f}x")
print(f"RFI channel {rfi_channel} ({freqs[rfi_channel]/1e6:.1f} MHz) was injected")

prms = prms_with_rfi  # Use RFI-weighted result for downstream plots

In [ ]:
vis_baseline = cal.simulate(prms_baseline)
vis_rfi_fit  = cal.simulate(prms)

loss_baseline_check = cal.calc_loss(prms_baseline)
loss_rfi_fit_check  = cal.calc_loss(prms)
print(f"\nContaminated data loss comparison:")
print(f"  Baseline (no RFI weighting):  {loss_baseline_check:.4e}")
print(f"  RFI-weighted fit:              {loss_rfi_fit_check:.4e}")
print(f"  Improvement:                   {loss_baseline_check / loss_rfi_fit_check:.2f}x")

# Compare residuals on contaminated data (vis_with_rfi)
resid_baseline = jnp.abs(vis_with_rfi - vis_baseline)
resid_rfi_fit  = jnp.abs(vis_with_rfi - vis_rfi_fit)

print(f"\nResidual power (on contaminated data):")
print(f"  Baseline RMS (all freqs):         {float(resid_baseline.mean()):.4e}")
print(f"  RFI-weighted RMS (all freqs):     {float(resid_rfi_fit.mean()):.4e}")
print(f"  Baseline RMS (RFI channel {rfi_channel}):  {float(resid_baseline[:, rfi_channel, :].mean()):.4e}")
print(f"  RFI-weighted RMS (RFI channel {rfi_channel}): {float(resid_rfi_fit[:, rfi_channel, :].mean()):.4e}")

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(13, 2.8 * len(bl_indices)), sharex=True)
fig.suptitle(
    f"Fit Comparison on RFI-Contaminated Data (channel {rfi_channel})"
    f"  —  Baseline loss {loss_baseline:.2e} → RFI-weighted loss {loss_with_rfi:.2e}",
    fontsize=11,
)
for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))

    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_with_rfi)[t_show, :, bi],         "k-",  lw=1.5, label="Contaminated data")
    ax.semilogy(freq_mhz, jnp.abs(vis_with_rfi - vis_baseline)[t_show, :, bi], 
                "b--", lw=1.2, label="Baseline residual", alpha=0.7)
    ax.semilogy(freq_mhz, jnp.abs(vis_with_rfi - vis_rfi_fit)[t_show, :, bi],  
                "r--", lw=1.5, label="RFI-weighted residual")
    ax.axvline(freqs[rfi_channel]/1e6, color='orange', linestyle=':', lw=2, label=f"RFI ch{rfi_channel}")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\n|V|")
    if row == 0: ax.legend(fontsize=8, loc='upper right')
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_with_rfi)[t_show, :, bi],         "k-",  lw=1.5, label="Contaminated data")
    ax.plot(freq_mhz, jnp.angle(vis_baseline)[t_show, :, bi], 
            "b--", lw=1.2, label="Baseline model", alpha=0.7)
    ax.plot(freq_mhz, jnp.angle(vis_rfi_fit)[t_show, :, bi],  
            "r--", lw=1.5, label="RFI-weighted model")
    ax.axvline(freqs[rfi_channel]/1e6, color='orange', linestyle=':', lw=2, label=f"RFI ch{rfi_channel}")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\narg(V) (rad)")
    if row == 0: ax.legend(fontsize=8, loc='upper right')
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

plt.tight_layout(); plt.show()

# Show per-channel residual power comparison
print("\nPer-channel RMS residual (averaged over time and baseline):")
resid_baseline_f = resid_baseline.mean(axis=(0, 2))  # Average over time and baseline
resid_rfi_fit_f  = resid_rfi_fit.mean(axis=(0, 2))
resid_data_f     = jnp.abs(vis_with_rfi).mean(axis=(0, 2))

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(freq_mhz, resid_data_f,     "k-",  lw=2, label="Data (contaminated)")
ax.semilogy(freq_mhz, resid_baseline_f, "b--", lw=1.5, label="Baseline residual", alpha=0.7)
ax.semilogy(freq_mhz, resid_rfi_fit_f,  "r--", lw=1.5, label="RFI-weighted residual")
ax.axvline(freqs[rfi_channel]/1e6, color='orange', linestyle=':', lw=2, label=f"RFI injected (ch{rfi_channel})")
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("RMS Residual")
ax.set_title("Per-channel RMS residual comparison: baseline vs RFI-weighted fit")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
from newnucal.rfi import channel_chi2_statistic, smooth_frequency_statistic, score_to_soft_weights

# Compute what channel weights SHOULD be based on baseline residuals
# This demonstrates the RFI detection logic
print("\n" + "="*70)
print("RFI CHANNEL WEIGHT DIAGNOSTICS")
print("="*70)

# Compute chi2 statistics on baseline residuals (without RFI weighting)
resid_baseline_raw = vis_with_rfi - vis_baseline
inv_noise_var = np.ones((nfreq,), dtype=np.float32)  # Uniform noise assumption
chi2_baseline = channel_chi2_statistic(resid_baseline_raw, inv_noise_var=inv_noise_var)
chi2_f_baseline = jnp.median(chi2_baseline, axis=0)  # Median over time

# Compare with RFI-weighted fit residuals
resid_rfi_raw = vis_with_rfi - vis_rfi_fit
chi2_rfi = channel_chi2_statistic(resid_rfi_raw, inv_noise_var=inv_noise_var)
chi2_f_rfi = jnp.median(chi2_rfi, axis=0)  # Median over time

# Smooth the chi2 to suppress single-pixel spikes
chi2_f_baseline_smooth = smooth_frequency_statistic(chi2_f_baseline, window=3)
chi2_f_rfi_smooth = smooth_frequency_statistic(chi2_f_rfi, window=3)

# Convert chi2 scores to soft weights (higher chi2 → lower weight)
weights_from_baseline = score_to_soft_weights(
    chi2_f_baseline_smooth, 
    center=1.5, 
    slope=2.0, 
    min_weight=0.01, 
    max_weight=1.0
)

weights_from_rfi = score_to_soft_weights(
    chi2_f_rfi_smooth, 
    center=1.5, 
    slope=2.0, 
    min_weight=0.01, 
    max_weight=1.0
)

print(f"\nChannel {rfi_channel} ({freqs[rfi_channel]/1e6:.1f} MHz) — RFI injection channel:")
print(f"  Baseline chi2:        {float(chi2_f_baseline[rfi_channel]):.4f}")
print(f"  RFI-weighted chi2:    {float(chi2_f_rfi[rfi_channel]):.4f}")
print(f"  Weight (from baseline chi2): {float(weights_from_baseline[rfi_channel]):.4f}")
print(f"  Weight (from RFI-fit chi2):  {float(weights_from_rfi[rfi_channel]):.4f}")

# Find neighboring channels for comparison
print(f"\nNeighboring channels:")
for offset in [-2, -1, 1, 2]:
    ch = rfi_channel + offset
    if 0 <= ch < nfreq:
        print(f"  Channel {ch}: baseline weight = {float(weights_from_baseline[ch]):.4f}, "
              f"RFI-fit weight = {float(weights_from_rfi[ch]):.4f}")

# Plot: chi2 and derived weights
fig, axes = plt.subplots(2, 1, figsize=(13, 6))

# Top panel: chi2 statistics
ax = axes[0]
ax.semilogy(freq_mhz, chi2_f_baseline,         "b-",  lw=1.5, label="Baseline (no RFI fit)", alpha=0.5)
ax.semilogy(freq_mhz, chi2_f_baseline_smooth, "b--", lw=1.5, label="Baseline (smoothed)")
ax.semilogy(freq_mhz, chi2_f_rfi,         "r-",  lw=1.5, label="RFI-weighted fit", alpha=0.5)
ax.semilogy(freq_mhz, chi2_f_rfi_smooth, "r--", lw=1.5, label="RFI-weighted (smoothed)")
ax.axvline(freqs[rfi_channel]/1e6, color='orange', linestyle=':', lw=2.5, label=f"RFI channel {rfi_channel}")
ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.5)
ax.set_ylabel("Chi² statistic (per channel)")
ax.set_title("Per-channel chi² residual power: baseline vs RFI-weighted fit")
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)

# Bottom panel: derived soft weights
ax = axes[1]
ax.plot(freq_mhz, weights_from_baseline, "b--", lw=2, label="Weights from baseline chi²", marker="o", markersize=3)
ax.plot(freq_mhz, weights_from_rfi,      "r--", lw=2, label="Weights from RFI-fit chi²", marker="s", markersize=3)
ax.axvline(freqs[rfi_channel]/1e6, color='orange', linestyle=':', lw=2.5, label=f"RFI channel {rfi_channel}")
ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.5)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("Soft weight")
ax.set_ylim([0, 1.1])
ax.set_title("Derived soft channel weights: RFI channel should be heavily downweighted in baseline")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print(f"RFI channel {rfi_channel} exhibits high chi² in baseline residuals,")
print(f"indicating poor fit quality. The soft weight fitting approach should")
print(f"assign this channel a low weight (~0.01–0.2) to downweight its")
print(f"contribution to the loss during joint optimization.")
print(f"\nWith RFI weighting enabled, the fit reduces pressure on the RFI")
print(f"channel and can focus on recovering the uncontaminated sky/beam,")
print(f"resulting in better overall model quality despite the contamination.")

## 7. Recovered sky map

In [ ]:
# Reconstruct sky flux from DPSS coefficients at a reference frequency
ifreq_ref = np.argmin(np.abs(freqs - ref_freq))

sky_true_map = np.array(flux_true[:, ifreq_ref])
# Deproject: unsolved pixels keep perturbed values; mask zeroes them out
sky_rec_full = np.array(sky_basis.deproject(np.asarray(prms["sky_coeffs"])))[:, ifreq_ref]
sky_rec_map  = cal.pixel_mask * sky_rec_full
sky_diff_map = cal.pixel_mask * (sky_rec_full - sky_true_map)

vmax = np.percentile(sky_true_map[sky_mask], 99)
vmin = 0.0
diff_scale = np.percentile(np.abs(sky_diff_map[sky_mask]), 99)

fig = plt.figure(figsize=(12, 9))
healpy.mollview(sky_true_map, fig=fig, sub=(3, 1, 1),
                title=f"True sky  ({ref_freq/1e6:.0f} MHz)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")
healpy.mollview(sky_rec_map,  fig=fig, sub=(3, 1, 2),
                title="Recovered sky  (joint fit, solved pixels only)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")
healpy.mollview(sky_diff_map, fig=fig, sub=(3, 1, 3),
                title="Residual (recovered − true, solved pixels)", min=-diff_scale, max=diff_scale,
                unit="Jy/pix", cmap="RdBu_r")
plt.show()

## 8. Recovered beam map and spectra

In [ ]:
# Reconstruct beam spectra from DPSS coefficients: (npix_beam, nfreq)
beam_spec_true     = np.array(beam_basis.deproject(np.asarray(beam_coeffs_true)))
beam_spec_perturb  = np.array(beam_basis.deproject(np.asarray(beam_coeffs_perturbed)))
beam_spec_rec      = np.array(beam_basis.deproject(np.asarray(prms["beam_coeffs"])))

rms_before = float(np.sqrt(np.mean((beam_spec_perturb - beam_spec_true) ** 2)))
rms_after  = float(np.sqrt(np.mean((beam_spec_rec     - beam_spec_true) ** 2)))
rms_signal = float(np.sqrt(np.mean(beam_spec_true ** 2)))
print(f"Beam RMS error — before: {rms_before:.4e}  after: {rms_after:.4e}  "
      f"(signal: {rms_signal:.4e})")
print(f"  relative: {rms_before/rms_signal:.3f} → {rms_after/rms_signal:.3f}")

# --- HEALPix maps at reference frequency ---
ifreq_ref = np.argmin(np.abs(freqs - ref_freq))

vmax_b = np.percentile(beam_spec_true[:, ifreq_ref], 99)

fig = plt.figure(figsize=(12, 10))
healpy.orthview(beam_spec_true[:, ifreq_ref], rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 1),
                title=f"True beam  ({ref_freq/1e6:.0f} MHz)",
                min=0, max=vmax_b, cmap="inferno")
healpy.orthview(beam_spec_rec[:, ifreq_ref], rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 2),
                title="Recovered beam  (joint fit)",
                min=0, max=vmax_b, cmap="inferno")
diff_b = beam_spec_rec[:, ifreq_ref] - beam_spec_true[:, ifreq_ref]
dscale = np.percentile(np.abs(diff_b), 99)
healpy.orthview(diff_b, rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 3),
                title="Residual (recovered − true)",
                min=-dscale, max=dscale, cmap="RdBu_r")
plt.show()

# --- Spectra at the peak-beam pixel and a few others ---
peak_px = int(np.argmax(beam_spec_true[:, ifreq_ref]))
sample_pxs = [peak_px,
               int(np.argsort(beam_spec_true[:, ifreq_ref])[-beam_spec_true.shape[0]//4]),
               int(np.argsort(beam_spec_true[:, ifreq_ref])[-beam_spec_true.shape[0]//2])]

fig, axes = plt.subplots(1, len(sample_pxs), figsize=(13, 4), sharey=False)
fig.suptitle("Beam spectra at selected pixels: true vs. perturbed vs. recovered", fontsize=11)
for ax, px in zip(axes, sample_pxs):
    ax.plot(freq_mhz, beam_spec_true[px],    "k-",  lw=2,   label="True")
    ax.plot(freq_mhz, beam_spec_perturb[px], "b--", lw=1.2, label="Perturbed", alpha=0.7)
    ax.plot(freq_mhz, beam_spec_rec[px],     "r--", lw=1.5, label="Recovered")
    ax.set_title(f"pixel {px}")
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Beam amplitude")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()